# Re-evaluate the 2048-context results at 4096

Needs GPU T4 x2, Internet, and the `HF_TOKEN` secret ticked. The main pair's primary 121-task set
was first evaluated with a 2048-token context, while the sweeps, the extended set, and every other
model pair used 4096. Long failing histories can exceed 2048 tokens and get truncated, so this
notebook puts every primary-set number on the same 4096 context:

1. **Main pair, eval only** (no training, the checkpoints are on Hugging Face): base (seed 0) and
   sft_only / sft_early / self_distill / distilled for seeds 0-4. Saved as
   `primary4k_eval_<condition>.json`, task_set `unseen_tools_ctx4096`.
2. **Second pair, retrain + eval** (its checkpoints were not kept): base (seed 0) and sft_only for
   seeds 0-2, saved as `pair2c_<condition>.json`. Its KD conditions already ran at 4096.

Nothing old is overwritten. Finished jobs are skipped, so a stopped run can simply be repeated.

In [ ]:
SEEDS_MAIN = [0, 1, 2, 3, 4]
MAIN_CONDITIONS = ["sft_only", "sft_early", "self_distill", "distilled"]   # + base (seed 0)
SEEDS_PAIR2 = [0, 1, 2]
RUN_PAIR2 = True
MAX_MINUTES = 300          # per phase: stop starting new jobs after this long

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Data (needed by the second-pair retraining)

In [ ]:
run_module("adbench.data.prepare", "--config", "configs/data.yaml")

## 2. Check the token can write

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN is not set: tick it under Add-ons -> Secrets for this notebook, then run again.")
os.environ["ADBENCH_RUN_TAG"] = "v2-seed0"
from adbench import pipeline_state as ps
ps.check_upload()

## 3. Run (two GPU-pinned workers per phase; no teacher is needed anywhere here)

In [ ]:
import subprocess
import sys
import threading


def pump(proc, name):
    for line in proc.stdout:
        if "Loading weights" in line or "it/s]" in line:
            continue
        print(f"[{name}] {line.rstrip()}", flush=True)


def run_phase(module, jobs, extra_args, tag):
    workers = []
    for gpu in (0, 1):
        share = jobs[gpu::2]
        if not share:
            continue
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        proc = subprocess.Popen(
            [sys.executable, "-m", module, "--jobs", ",".join(share),
             "--work-dir", f"/kaggle/working/{tag}_gpu{gpu}", "--max-minutes", str(MAX_MINUTES), *extra_args],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
        )
        t = threading.Thread(target=pump, args=(proc, f"{tag}_gpu{gpu}"), daemon=True)
        t.start()
        workers.append((proc, t))
    for proc, t in workers:
        proc.wait()
        t.join(timeout=30)
    codes = [p.returncode for p, _ in workers]
    print(tag, "exit codes:", codes)
    if any(codes):
        raise RuntimeError(f"a {tag} worker failed; see the log above")


# condition-major so both GPUs get a similar mix of conditions
main_jobs = ["0:base"] + [f"{s}:{c}" for c in MAIN_CONDITIONS for s in SEEDS_MAIN]
print(f"### Phase 1: main pair, {len(main_jobs)} eval-only jobs ###")
run_phase("adbench.evaluation.ext_worker", main_jobs, ["--task-set", "primary"], "main4k")

if RUN_PAIR2:
    pair2_jobs = ["0:base"] + [f"{s}:sft_only" for s in SEEDS_PAIR2]
    print(f"### Phase 2: second pair, {len(pair2_jobs)} retrain+eval jobs ###")
    run_phase("adbench.evaluation.second_pair_worker", pair2_jobs, ["--rerun-4k"], "pair2c")

## Summary

In [ ]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/*_gpu*/results/*.json"):
    rows += json.load(open(path, encoding="utf-8"))
if rows:
    df = pd.DataFrame(rows)
    df["c35"] = df.chain_length.isin([3, 5])
    print(df.groupby(["task_set", "condition", "seed"]).success.mean().round(3))
    print(df[df.c35].groupby(["task_set", "condition"]).success.mean().round(3))
else:
    print("no result files in this session (everything was already on Hugging Face)")